# Lab 21-04 · Low-Noise Amplifiers

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import lna

db = lambda x: 10 * math.log10(x)


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


F0, GM, FT = 2.4e9, 0.04, 12e9
d = lna.design_ind_degen(F0, GM, FT)
print({k: f"{val:.4g}" for k, val in d.items()})

{'gm': '0.04', 'cgs': '5.305e-13', 'ls': '6.631e-10', 'lg': '7.626e-09', 'q_in': '1.25', 'gm_eff': '0.05'}


## Part 1 · The match

*Predict* the 10 dB return-loss band from the input loop's Q.

In [2]:
f = np.linspace(1e9, 4e9, 300001)
z = np.array([lna.zin_ind_degen(ff, GM, d["cgs"], d["ls"], d["lg"]) for ff in f])
rl = -20 * np.log10(abs((z - 50) / (z + 50)))
band = f[rl >= 10]
print(f"Re Z_in = {z.real.mean():.2f} ohm at every frequency; 10 dB band {band.min() / 1e9:.3f}-{band.max() / 1e9:.3f} GHz "
      f"({(band.max() - band.min()) / F0 * 100:.1f} %)")
plt.figure(); plt.plot(f / 1e9, rl); plt.axhline(10, ls="--"); plt.ylim(0, 40)
plt.xlabel("frequency (GHz)"); plt.ylabel("return loss (dB)"); plt.show()

Re Z_in = 50.00 ohm at every frequency; 10 dB band 2.101-2.741 GHz (26.7 %)


## Part 2 · Inductor Q

*Predict* the gate-inductor Q at which the noise figure reaches 1 dB.

In [3]:
def nf_q(q):
    dd, rg = lna.design_with_gate_q(F0, GM, FT, q)
    return db(lna.nf_ind_degen(1, GM, F0, FT, rg=rg))


qs = np.linspace(5, 100, 200)
plt.figure(); plt.plot(qs, [nf_q(q) for q in qs]); plt.axhline(1, ls="--")
plt.xlabel("gate inductor Q"); plt.ylabel("NF (dB)"); plt.show()
print(f"NF at Q = 5: {nf_q(5):.2f} dB; 1 dB at Q = {optimize.brentq(lambda q: nf_q(q) - 1, 5, 100):.1f}")

NF at Q = 5: 2.19 dB; 1 dB at Q = 15.3


## Part 3 · Q, noise and linearity

*Predict* how the noise figure and the IIP3 shift change as $f_T$ doubles.

In [4]:
fts = np.linspace(6e9, 48e9, 50)
nfs = [db(lna.nf_ind_degen(1, GM, F0, ft)) for ft in fts]
shift = [lna.iip3_shift_db(lna.design_ind_degen(F0, GM, ft)["q_in"]) for ft in fts]
fig, ax = plt.subplots(); ax.plot(fts / 1e9, nfs); ax.set_xlabel("f_T (GHz)"); ax.set_ylabel("NF (dB)")
ax2 = ax.twinx(); ax2.plot(fts / 1e9, shift, "C1"); ax2.set_ylabel("IIP3 penalty 20 log Q_in (dB)"); plt.show()
for ft in (12e9, 24e9):
    print(f"f_T {ft / 1e9:.0f} GHz: NF {db(lna.nf_ind_degen(1, GM, F0, ft)):.3f} dB, "
          f"IIP3 penalty {lna.iip3_shift_db(lna.design_ind_degen(F0, GM, ft)['q_in']):.2f} dB")

f_T 12 GHz: NF 0.334 dB, IIP3 penalty 1.94 dB
f_T 24 GHz: NF 0.086 dB, IIP3 penalty 7.96 dB


## Part 4 · Wideband: noise cancelling

*Predict* the noise figure at $g_{m2}$ = 50, 100 and 200 mS, and how it compares with a common-gate stage.

In [5]:
g2 = np.linspace(0.02, 0.3, 100)
plt.figure(); plt.plot(g2 * 1e3, [db(lna.nf_noise_cancelling(2 / 3, g, 200)) for g in g2], label="noise cancelling")
plt.axhline(db(lna.nf_common_gate(2 / 3, 0.02)), ls="--", label="common gate")
plt.xlabel("g_m2 (mS)"); plt.ylabel("NF (dB)"); plt.legend(); plt.show()
for g in (0.05, 0.1, 0.2):
    print(f"g_m2 = {g * 1e3:.0f} mS: NF {db(lna.nf_noise_cancelling(2 / 3, g, 200)):.2f} dB")


def nf_r1(r1, x=2 / 3, gm1=0.02, gm2=0.1, r2=200.0, rs=50.0):
    a = (r1 / rs + gm2 * r2) / 2
    num = x * gm1 * ((gm2 * r2 * rs - r1) / 2) ** 2 + x * gm2 * r2 ** 2 + r1 + r2
    return db(1 + num / (a * a * rs))


print([f"R1 {r}: {nf_r1(r):.2f} dB" for r in (900, 1000, 1100)])

g_m2 = 50 mS: NF 1.48 dB
g_m2 = 100 mS: NF 0.77 dB
g_m2 = 200 mS: NF 0.39 dB
['R1 900: 0.83 dB', 'R1 1000: 0.77 dB', 'R1 1100: 0.72 dB']


## Part 5 · ngspice

*Predict* the four noise figures before running.

In [6]:
s = ngspice("lna_noise.cir")
print({k: round(s[k], 3) for k in ("rina", "nfa", "nfb", "nfc", "nfd")})

{'rina': 50.0, 'nfa': 0.334, 'nfb': 1.011, 'nfc': 2.218, 'nfd': 0.768}
